# MySQL Game Log ETL — Public Demonstration

This notebook is a **sanitized educational reconstruction**, not the original proprietary implementation. Uses synthetic sample data by default. For an authorized MySQL database, configure environment variables and set `USE_SAMPLE_DATA = False`. No real credentials, identifiers, or outputs are included.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from datetime import datetime, timedelta

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
USE_SAMPLE_DATA = True
START_DATE = "2025-09-01"
END_DATE = "2025-09-30"
TIMEZONE_LABEL = "UTC+08"  # Label only; no timezone conversion is performed

## Extract
Date filters use a half-open interval (`>= start`, `< next day`) to avoid missing fractional-second timestamps.

In [ ]:
start = pd.Timestamp(START_DATE)
end_exclusive = pd.Timestamp(END_DATE) + pd.Timedelta(days=1)

if USE_SAMPLE_DATA:
    raw = pd.read_csv(ROOT / "sample_data" / "sample_game_logs.csv", parse_dates=["end_at"])
else:
    from dotenv import load_dotenv
    from sqlalchemy import create_engine, text
    from sqlalchemy.engine import URL
    load_dotenv(ROOT / ".env")
    required = ["DB_HOST", "DB_USER", "DB_PASSWORD", "DB_NAME"]
    missing = [key for key in required if not os.getenv(key)]
    if missing:
        raise RuntimeError("Missing required database settings: " + ", ".join(missing))
    url = URL.create("mysql+pymysql", username=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"], host=os.environ["DB_HOST"], port=int(os.getenv("DB_PORT", "3306")), database=os.environ["DB_NAME"])
    engine = create_engine(url, pool_pre_ping=True)
    # Illustrative schema only: adapt to an authorized database.
    query = text("SELECT provider_id, end_at, bet_amount, payout_amount FROM game_logs WHERE end_at >= :start AND end_at < :end")
    try:
        with engine.connect() as conn:
            raw = pd.read_sql_query(query, conn, params={"start": start.to_pydatetime(), "end": end_exclusive.to_pydatetime()})
    finally:
        engine.dispose()

## Transform
Validate required columns, numeric amounts and timestamps. Aggregate by provider without selecting an ungrouped timestamp.

In [ ]:
required_columns = {"provider_id", "end_at", "bet_amount", "payout_amount"}
if not required_columns.issubset(raw.columns):
    raise ValueError("Missing expected columns: " + str(sorted(required_columns - set(raw.columns))))
df = raw.copy()
df["end_at"] = pd.to_datetime(df["end_at"], errors="raise")
for col in ("bet_amount", "payout_amount"):
    df[col] = pd.to_numeric(df[col], errors="raise")
if df[["provider_id", "end_at", "bet_amount", "payout_amount"]].isna().any().any():
    raise ValueError("Null values detected in required fields")
df = df.loc[(df["end_at"] >= start) & (df["end_at"] < end_exclusive)]
report = df.groupby("provider_id", as_index=False).agg(Bet=("bet_amount", "sum"), Payout=("payout_amount", "sum"), Records=("provider_id", "size"))
print(f"Validated {len(df)} sample records across {len(report)} providers")
report

## Load
Write a new Excel report. The output folder is ignored by Git.

In [ ]:
filename = f"game_log_report_{start:%Y%m%d}_{pd.Timestamp(END_DATE):%Y%m%d}.xlsx"
path = OUTPUT_DIR / filename
report.to_excel(path, index=False, engine="openpyxl")
print("Report saved:", path)